# Gold Clinical Analytics

## Purpose

I use this notebook as one of the transformation sources for the Gold Lakeflow
pipeline.

I create a patient-level clinical analytical dataset from the validated FHIR
Silver datasets.

### Sources

- `health_insurance.silver.fhir_patient`
- `health_insurance.silver.fhir_condition`
- `health_insurance.silver.fhir_encounter`

### Pipeline target

`health_insurance.gold.patient_clinical_summary`

### Grain

One row represents one FHIR patient.

### Modeling approach

FHIR Patient, Condition, and Encounter are separate clinical entities with
different grains.

A patient can have:

- many Conditions
- many Encounters

I therefore aggregate Conditions and Encounters independently to the Patient
grain before joining them to the Patient dataset.

This prevents row multiplication that would occur if I directly joined all
three detailed datasets together.

I keep this clinical model separate from the Kaggle Claims dimensional model
because I do not have a verified crosswalk connecting the two patient
identifier systems.

In [0]:
#  importing the Lakeflow API and Spark functions used by the clinical model.

from pyspark import pipelines as dp
from pyspark.sql import functions as F

CATALOG = "health_insurance"

PATIENT_SOURCE = f"{CATALOG}.silver.fhir_patient"
CONDITION_SOURCE = f"{CATALOG}.silver.fhir_condition"
ENCOUNTER_SOURCE = f"{CATALOG}.silver.fhir_encounter"

## Patient condition metrics

I aggregate the FHIR Condition dataset to one row per Patient before joining it
to the Patient dataset.

I calculate:

- total Conditions
- distinct condition codes
- active Conditions
- resolved Conditions
- confirmed Conditions
- first recorded condition onset
- latest recorded condition onset

These metrics describe the clinical condition history available in the FHIR
source without changing the final one-row-per-patient grain.

In [0]:
# aggregating FHIR Conditions to the Patient grain.

def build_condition_metrics(condition_df):

    return (
        condition_df

        .groupBy(
            "patient_id"
        )

        .agg(
            F.count("*").alias(
                "total_conditions"
            ),

            F.countDistinct(
                "condition_code"
            ).alias(
                "distinct_condition_codes"
            ),

            F.sum(
                F.when(
                    F.col("clinical_status") == "ACTIVE",
                    1
                ).otherwise(0)
            ).alias(
                "active_conditions"
            ),

            F.sum(
                F.when(
                    F.col("clinical_status") == "RESOLVED",
                    1
                ).otherwise(0)
            ).alias(
                "resolved_conditions"
            ),

            F.sum(
                F.when(
                    F.col("verification_status") == "CONFIRMED",
                    1
                ).otherwise(0)
            ).alias(
                "confirmed_conditions"
            ),

            F.min(
                "onset_datetime"
            ).alias(
                "first_condition_onset"
            ),

            F.max(
                "onset_datetime"
            ).alias(
                "latest_condition_onset"
            )
        )
    )

## Patient encounter metrics

I aggregate FHIR Encounters independently to the Patient grain.

I calculate:

- total Encounters
- ambulatory Encounters
- emergency Encounters
- inpatient Encounters
- distinct organizations
- distinct practitioners
- first Encounter
- latest Encounter
- average Encounter duration
- total recorded Encounter duration

I calculate Encounter duration only when both the start and end timestamps are
available.

In [0]:
# aggregating FHIR Encounters to the Patient grain.

def build_encounter_metrics(encounter_df):

    encounter_with_duration_df = (
        encounter_df

        .withColumn(
            "_encounter_duration_hours",

            F.when(
                F.col("start_datetime").isNotNull()
                &
                F.col("end_datetime").isNotNull(),

                (
                    F.unix_timestamp("end_datetime")
                    -
                    F.unix_timestamp("start_datetime")
                ) / 3600
            )
        )
    )

    return (
        encounter_with_duration_df

        .groupBy(
            "patient_id"
        )

        .agg(
            F.count("*").alias(
                "total_encounters"
            ),

            F.sum(
                F.when(
                    F.col("encounter_class") == "AMB",
                    1
                ).otherwise(0)
            ).alias(
                "ambulatory_encounters"
            ),

            F.sum(
                F.when(
                    F.col("encounter_class") == "EMER",
                    1
                ).otherwise(0)
            ).alias(
                "emergency_encounters"
            ),

            F.sum(
                F.when(
                    F.col("encounter_class") == "IMP",
                    1
                ).otherwise(0)
            ).alias(
                "inpatient_encounters"
            ),

            F.countDistinct(
                "organization_reference"
            ).alias(
                "distinct_organizations"
            ),

            F.countDistinct(
                "practitioner_reference"
            ).alias(
                "distinct_practitioners"
            ),

            F.min(
                "start_datetime"
            ).alias(
                "first_encounter_start"
            ),

            F.max(
                "start_datetime"
            ).alias(
                "latest_encounter_start"
            ),

            F.max(
                "end_datetime"
            ).alias(
                "latest_encounter_end"
            ),

            F.avg(
                "_encounter_duration_hours"
            ).alias(
                "average_encounter_duration_hours"
            ),

            F.sum(
                "_encounter_duration_hours"
            ).alias(
                "total_encounter_duration_hours"
            )
        )
    )

## Patient clinical summary

I use FHIR Patient as the base entity because the final analytical grain is one
row per Patient.

I left join the aggregated Condition and Encounter metrics so Patients remain
in the Gold dataset even when they have no recorded Conditions or Encounters.

I replace missing count metrics with zero while preserving missing clinical
timestamps as null values because a missing date has different semantics from
a count of zero.

In [0]:
# defining the pipeline-managed Patient clinical summary.

@dp.materialized_view(
    name="patient_clinical_summary",
    comment="FHIR patient-level clinical summary combining condition and encounter history."
)
def patient_clinical_summary():

    patient_df = spark.read.table(
        PATIENT_SOURCE
    )

    condition_df = spark.read.table(
        CONDITION_SOURCE
    )

    encounter_df = spark.read.table(
        ENCOUNTER_SOURCE
    )

    condition_metrics_df = (
        build_condition_metrics(
            condition_df
        )
    )

    encounter_metrics_df = (
        build_encounter_metrics(
            encounter_df
        )
    )

    clinical_df = (
        patient_df

        .join(
            condition_metrics_df,
            on="patient_id",
            how="left"
        )

        .join(
            encounter_metrics_df,
            on="patient_id",
            how="left"
        )
    )

    return (
        clinical_df

        # replacing missing aggregate counts with zero.

        .fillna(
            0,
            subset=[
                "total_conditions",
                "distinct_condition_codes",
                "active_conditions",
                "resolved_conditions",
                "confirmed_conditions",
                "total_encounters",
                "ambulatory_encounters",
                "emergency_encounters",
                "inpatient_encounters",
                "distinct_organizations",
                "distinct_practitioners"
            ]
        )

        # deriving simple Patient-level clinical activity indicators.

        .withColumn(
            "has_active_conditions",
            F.col("active_conditions") > 0
        )

        .withColumn(
            "has_encounter_history",
            F.col("total_encounters") > 0
        )

        #  rounding duration metrics for reporting.

        .withColumn(
            "average_encounter_duration_hours",
            F.round(
                F.col(
                    "average_encounter_duration_hours"
                ),
                2
            )
        )

        .withColumn(
            "total_encounter_duration_hours",
            F.round(
                F.col(
                    "total_encounter_duration_hours"
                ),
                2
            )
        )

        .select(
            # Patient identity
            "patient_id",
            "medical_record_number",

            # Patient demographics
            "given_name",
            "family_name",
            "gender",
            "birth_date",
            "age",
            "age_group",
            "phone",
            "city",
            "state",
            "postal_code",
            "country",

            # Condition metrics
            "total_conditions",
            "distinct_condition_codes",
            "active_conditions",
            "resolved_conditions",
            "confirmed_conditions",
            "first_condition_onset",
            "latest_condition_onset",
            "has_active_conditions",

            # Encounter metrics
            "total_encounters",
            "ambulatory_encounters",
            "emergency_encounters",
            "inpatient_encounters",
            "distinct_organizations",
            "distinct_practitioners",
            "first_encounter_start",
            "latest_encounter_start",
            "latest_encounter_end",
            "average_encounter_duration_hours",
            "total_encounter_duration_hours",
            "has_encounter_history",

            # Source lineage
            "_source_system",
            "_ingested_at"
        )

        .withColumn(
            "_gold_transformed_at",
            F.current_timestamp()
        )
    )

## Clinical analytical output

When this notebook is added to the Gold Lakeflow pipeline, it defines:

`health_insurance.gold.patient_clinical_summary`

### Grain

One row per FHIR Patient.

### Patient attributes

The model retains selected demographic and identification fields from the
validated FHIR Patient dataset.

### Condition metrics

- total Conditions
- distinct condition codes
- active Conditions
- resolved Conditions
- confirmed Conditions
- first condition onset
- latest condition onset
- active-condition indicator

### Encounter metrics

- total Encounters
- ambulatory Encounters
- emergency Encounters
- inpatient Encounters
- distinct organizations
- distinct practitioners
- first Encounter
- latest Encounter
- average Encounter duration
- total Encounter duration
- Encounter-history indicator

### Join strategy

I aggregate Conditions and Encounters independently to Patient grain before
joining them to FHIR Patient.

This prevents many-to-many row multiplication and preserves the intended
one-row-per-patient analytical grain.

### Domain separation

I do not join this dataset to the Kaggle Claims Patient dimension because the
two sources do not provide a verified identifier crosswalk.

The Claims and FHIR analytical models therefore remain separate, defensible
business domains within the same lakehouse.

I do not manually persist or execute this dataset in this notebook. Lakeflow
will manage the materialized view when the Gold pipeline is eventually
deployed and run.